# Inspect DT-GFN tree results

Interactive front-end for `aggregate_treeclass_results.py`: pick a dataset and
whether you want **training** or **debug** runs, and see the per-configuration
tables (mean ± std over dataset splits, newest launch first) without re-reading
anything from disk on every change.

**How to run**
1. Start jupyter with the gflownet venv active (`module load python/3.10` and
   `source ~/scratch/venvs/gflownet-env/bin/activate`), e.g.
   `jupyter notebook` on a compute/interactive node, or use this venv as the
   kernel in VS Code.
2. Run the *Load data* cell once (the wandb scan takes ~1 min; set
   `LOAD_WANDB = False` to skip it).
3. Use the widget UI below, and `hash2config("<hash>")` to see the full
   hyperparameter config behind any group hash.


In [1]:
# ------------------------------ Load data -----------------------------------
# Reads every eval_results.json under ROOT and (optionally) all runs of the
# wandb projects. Re-run this cell to refresh.
from pathlib import Path

import pandas as pd
from IPython.display import display
from omegaconf import OmegaConf

from gflownet.envs.tree.helpers_for_experiments import (
    aggregate_treeclass_results as agg,
)

ROOT = agg.default_root()      # e.g. $SCRATCH/gflownet-logs; set your own Path(...) here
LOAD_WANDB = True              # False -> disk (eval_results.json) only
WANDB_NAME_PREFIX = None       # e.g. "MAGIC_STAB" to only scan those runs

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

records = {"eval": agg.collect_eval_runs(Path(ROOT))}
if LOAD_WANDB:
    records["wandb"] = agg.collect_wandb_runs(
        agg.WANDB_ENTITY, agg.WANDB_PROJECTS, name_prefix=WANDB_NAME_PREFIX
    )

print({src: len(recs) for src, recs in records.items()})

[WARN] duplicate eval runs for diabetes split 1 config 14e1f073 (TRFM_REG_CPU_PROF2_diabetes1_depth5_steps500_lr0.01_trfm_seed0_490ee6e9 / TRFM_REG_CPU_PROF_16T_diabetes1_depth5_steps500_lr0.01_trfm_seed0_490ee6e9); keeping the one with the newest eval_results.json.


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/mila/a/arnit/.netrc.


[INFO] wandb: scanning 578 runs in alex-hg/dt-gfn_classification ...
[INFO] wandb: scanning 212 runs in alex-hg/dt-gfn_regression ...
[WARN] duplicate wandb runs for iris split 3 config 9d5d7037 (transformer_policy_default_depth1_iris1 / trfm_iris_depth1_split3); keeping the run that progressed furthest (ties: latest launch).
{'eval': 511, 'wandb': 676}


In [2]:
# ------------------------------ Widget UI -----------------------------------
import ipywidgets as widgets

_all_recs = [r for recs in records.values() for r in recs]
_datasets = sorted({r["dataset"] for r in _all_recs})

_dataset_w = widgets.Dropdown(options=_datasets, description="dataset")
_section_w = widgets.ToggleButtons(
    options=[("training", False), ("debug", True)], description="runs"
)
_source_w = widgets.ToggleButtons(
    options=[s for s in ("eval", "wandb") if s in records] + ["both"],
    value="both" if len(records) > 1 else list(records)[0],
    description="source",
)
_minsplits_w = widgets.IntSlider(
    value=agg.MIN_SPLITS_DEFAULT, min=1, max=10, description="min splits"
)


def show_tables(dataset, debug, source, min_splits):
    sources = list(records) if source == "both" else [source]
    shown_anything = False
    for src in sources:
        recs = [
            r
            for r in records.get(src, [])
            if r["dataset"] == dataset and r["debug"] == debug
        ]
        for task in sorted({r["task"] for r in recs}):
            task_recs = [r for r in recs if r["task"] == task]
            metric_map = agg.EVAL_METRICS if src == "eval" else agg.WANDB_METRICS
            df, n_hidden = agg.build_table(
                task_recs, metric_map[task], src, min_splits
            )
            label = (
                "eval_results.json (final evaluation)"
                if src == "eval"
                else "wandb (last logged values)"
            )
            print(f"--- {dataset} ({task}) | {label} ---")
            if len(df):
                display(df)
                shown_anything = True
            if n_hidden:
                print(f"[note] {n_hidden} configuration(s) hidden (< {min_splits} splits)")
    if not shown_anything:
        print("No matching runs (try min splits = 1 or the other section/source).")


widgets.interact(
    show_tables,
    dataset=_dataset_w,
    debug=_section_w,
    source=_source_w,
    min_splits=_minsplits_w,
);

interactive(children=(Dropdown(description='dataset', options=('breast_cancer', 'concrete', 'credit_quantile',…

In [4]:
# ------------------------------ hash2config ---------------------------------
def hash2config(group_hash: str, source: str = None):
    """Display the full hyperparameter config behind a group hash.

    The config shown is exactly what was hashed into the group id: the run's
    resolved hydra config WITHOUT run-specific keys (dataset split path,
    logger/run name, user section, slurm_job_id, ...). Accepts a hash prefix.
    """
    sources = [source] if source else list(records)
    for src in sources:
        for rec in records.get(src, []):
            if rec["hash"].startswith(group_hash):
                print(
                    f"# config {rec['hash']} | {rec['dataset']} ({rec['task']}) "
                    f"| campaign {rec['campaign']} | e.g. run {rec['run_name']}\n"
                )
                print(OmegaConf.to_yaml(OmegaConf.create(rec["config"])))
                return
    print(f"No run with config hash starting with {group_hash!r} loaded.")


# Example -- paste any hash from the tables above:
hash2config("cd969423")

# config cd969423 | diabetes (regression) | campaign REG_NIG_GRID | e.g. run REG_NIG_GRID_diabetes1_depth5_steps10000_lr0.001_mlp_seed0_68fc75ee

device: cpu
float_precision: 32
n_samples: 1000
seed: 0
env:
  _target_: gflownet.envs.tree.regression_tree.RegressionTree
  env_id: env
  fixed_distr_params: null
  random_distr_params: null
  skip_mask_check: false
  conditional: false
  continuous: false
  max_depth: 5
  scale_y: true
  rescale_thresholds: true
  node_kwargs:
    features: null
    cube_kwargs:
      n_comp: 3
      beta_params_min: 1
      beta_params_max: 100
      min_incr: 0.99
      fixed_distr_params:
        beta_weights: 1
        beta_alpha: 10
        beta_beta: 10
        bernoulli_eos_prob: 0.1
        bernoulli_bts_prob: 0.1
      random_distr_params:
        beta_weights: 1
        beta_alpha: 1
        beta_beta: 1
        bernoulli_eos_prob: 0.1
        bernoulli_bts_prob: 0.1
gflownet:
  _target_: gflownet.gflownet.GFlowNetAgent
  seed: 0
  optimizer:
    

**Caveats**
- The tables are built from the records loaded in the first cell; re-run it to
  pick up new runs or fresh wandb values.
- wandb metrics are the *last logged* values: for `state=running` rows they
  are mid-training numbers, not final results (check `last_step`).
- `mean_n_decisionnodes` counts decision (internal) nodes only;
  classification `model_size_*` counts decision nodes **plus** leaves.
- `hash2config` needs at least one loaded run with that hash; it shows the
  group-defining config (run-identity keys removed), not the literal
  `.hydra/config.yaml` of a specific run.
